# Heatmap - issue # 47

Considering 12 months of data:

For simplicity, extract data from the database to a csv file for visualisation.

Visualisation

- x axis: month, categorical
- y axis: country, categorical
- colour: number of mentions, integer
- pallet: from matplotlib
- package: Bokeh

Rationale for using Bokeh based on blog post

- It lets you select date ranges
- We can update multiple graphs based on the same data (changes).
- We can also embed it in an HTML.
- Easy to include a legend for the scale
- Can be added to a web app.

Alternatively, a very simple but less customisable package would be Holoviews which infers the right graph type from the data itself but it is less mature.

In [1]:
import curl_cffi
import pandas as pd
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_alternative_country_names"
response = curl_cffi.get(url, impersonate="chrome")
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all(class_="wikitable")

data = []
for table in tables:
    for row in table.find_all("tr")[1:]: # ignores the column names row
        cell_data = row.find_all("td")
        if len(cell_data) == 3: # only process when there are 3 columns
            code, name, names = cell_data
            country_name = name.find("a")["title"].strip()
            other_names = "|".join(
                [x.text.strip() for x in names.find_all("b")]
                + [x.text.strip() for x in names.find_all("i")]
            )
            data.append((code.text, country_name, other_names))

country_df = pd.DataFrame(data, columns=["code", "name", "alternative_names"])
country_df

,code,name,alternative_names
0,,Abkhazia,Republic of Abkhazia|Apsny|Aphsny Axwynthkharr...
1,AFG,Afghanistan,Islamic Emirate of Afghanistan|Da Afġānistān I...
2,ALB,Albania,Republic of Albania|Republika e Shqipërisë|Arn...
3,DZA,Algeria,People's Democratic Republic of Algeria|al-Jaz...
4,AND,Andorra,Principality of Andorra|Principat d'Andorra|Pr...
...,...,...,...
195,VNM,Vietnam,Socialist Republic of Vietnam|An Nam|Champa|Đạ...
196,ESH,Sahrawi Arab Democratic Republic,Sahrawi Arab Democratic Republic|Saharawi Arab...
197,YEM,Yemen,Republic of Yemen|ٱلْجُمْهُورِيَّةُ ٱلْيَمَنِي...
198,ZMB,Zambia,Republic of Zambia|Northern Rhodesia


In [2]:
# Count country mentions by code, name, or alternative names (separated by "|")

def count_country_mentions(section, country_df):
    """Count mentions of each country in a section text."""
    counts = {}

    for _, row in country_df.iterrows():
        country_key = row['name']  # Use name as the key
        code = row['code'].strip() if pd.notna(row['code']) else ""
        name = row['name'].strip() if pd.notna(row['name']) else ""
        alt_names = row['alternative_names'].split("|") if pd.notna(row['alternative_names']) else []

        # Check if section contains code, name, or any alternative name
        found = False
        if code and code.lower() in section.lower():
            found = True
        elif name and name.lower() in section.lower():
            found = True
        else:
            for alt_name in alt_names:
                if alt_name.strip() and alt_name.strip().lower() in section.lower():
                    found = True
                    break

        if found:
            counts[country_key] = counts.get(country_key, 0) + 1

    return counts



In [5]:
import shutil

zip_path = r"../../data/processed/database/briefings.db.zip"
extract_dir = r"../../data/processed/database"
db_path = os.path.join(extract_dir, "briefings.db")

# Remove the directory if it exists
if os.path.exists(db_path) and os.path.isdir(db_path):
    shutil.rmtree(db_path)
    print("✓ Removed briefings.db directory")

# Extract the zip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction complete!")
print(f"Database size: {os.path.getsize(db_path)} bytes")

✓ Removed briefings.db directory
✓ Extraction complete!
Database size: 35958784 bytes


In [ ]:
import sqlite3
from pathlib import Path

conn = sqlite3.connect(db_path)

query = """
SELECT s.section_id, s.briefing_id, s.section_order, s.title, s.text, b.briefing_date
FROM section s
JOIN briefing b ON s.briefing_id = b.briefing_id
"""
sections_df = pd.read_sql_query(query, conn)
conn.close()

# Extract month from briefing_date
sections_df['month'] = pd.to_datetime(sections_df['briefing_date']).dt.to_period('M')

# Count country mentions for each section
all_country_counts = {}

for _, row in sections_df.iterrows():
    section_text = row['text']
    section_month = row['month']

    # Count mentions for this section
    counts = count_country_mentions(section_text, country_df)

    # Aggregate by month
    for country, count in counts.items():
        key = (section_month, country)
        all_country_counts[key] = all_country_counts.get(key, 0) + count

# Convert to DataFrame for heatmap
heatmap_data = []
for (month, country), count in all_country_counts.items():
    heatmap_data.append({
        'month': str(month),
        'country': country,
        'count': count
    })

heatmap_df = pd.DataFrame(heatmap_data)
heatmap_df